# AgroNova LLM untuk bot WhatsApp (Colab)

Menjalankan model open source (default `qwen3:4b`) di GPU Colab untuk menjawab pertanyaan di bot
WhatsApp (`apps/whatsapp-bot`). Bot dan API tetap jalan di laptop; hanya model yang "pinjam" GPU Colab.

1. Runtime → Change runtime type → **T4 GPU**
2. Atur `MODEL` dan `JALANKAN_VLM` di sel pertama
3. **Run all** (pertama kali ±5 menit; tambah ±5-10 menit jika VLM ikut dijalankan)
4. Salin baris `WA_LLM_*` dari output sel terakhir ke `apps/whatsapp-bot/.env`, lalu restart bot.
   Jika `JALANKAN_VLM = True`, salin juga baris `INFERENCE_*` (dicetak oleh bagian VLM) ke `apps/api/.env`.

Catatan:
- **Ollama tidak punya autentikasi**, sedangkan URL tunnel bisa diakses siapa pun. Karena itu Ollama hanya
  mendengar di `127.0.0.1`, dan yang dibuka lewat tunnel adalah proxy kecil yang mewajibkan
  `WA_LLM_API_KEY` dan hanya meneruskan `POST /v1/chat/completions`.
- **Colab gratis biasanya hanya memberi satu sesi GPU sekaligus.** Jika website juga butuh deteksi stres,
  pakai `JALANKAN_VLM = True` di sini, bukan notebook VLM di tab lain. VLM dijalankan lebih dulu agar
  mendapat memori GPU; sisanya untuk LLM.
- **Hanya untuk demo.** Sesi bisa mati sewaktu-waktu dan URL berubah tiap run. Saat LLM tidak bisa
  dihubungi, bot tetap mengirim warning dan membalas pertanyaan dengan ringkasan status tetap.

In [ ]:
MODEL = "qwen3:4b"     # "qwen3:8b" hanya jika sel pemeriksaan GPU menunjukkan memori cukup
JALANKAN_VLM = False   # True = jalankan juga deteksi stres (VLM) di GPU yang sama, lebih dulu
LLM_PORT = 8081        # proxy ber-API key; Ollama sendiri hanya di 127.0.0.1:11434
# Notebook VLM dijalankan apa adanya dari repo, jadi langkahnya selalu sama dengan aslinya.
VLM_NOTEBOOK = ("https://raw.githubusercontent.com/WANGDELL30/agronova-poultry-dashboard/"
                "main/apps/inference/colab/run_inference_colab.ipynb")

import os, shutil, subprocess
# Tanpa GPU, Ollama diam-diam jalan di CPU dan setiap jawaban butuh puluhan detik.
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Tidak ada GPU. Runtime -> Change runtime type -> T4 GPU, lalu Run all lagi.")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
HOME = os.getcwd()

In [ ]:
if JALANKAN_VLM:
    import json, urllib.request
    vlm_notebook = json.load(urllib.request.urlopen(VLM_NOTEBOOK))
    for cell in vlm_notebook["cells"]:
        if cell["cell_type"] != "code":
            continue
        if not get_ipython().run_cell("".join(cell["source"])).success:
            raise RuntimeError("Bagian VLM gagal; lihat error di atas.")
    os.chdir(HOME)  # notebook VLM pindah ke repo/apps/inference
    try:
        vlm_status = json.load(urllib.request.urlopen("http://localhost:8000/health"))["status"]
    except Exception as error:
        vlm_status = f"tidak dapat dihubungi ({error})"
    if vlm_status != "ready":
        print(f"PERINGATAN: VLM belum siap ({vlm_status}). LLM tetap dijalankan.")
else:
    print("VLM dilewati (JALANKAN_VLM = False).")

In [ ]:
import time, urllib.request
# Matikan sisa run sebelumnya (hanya proses LLM; VLM tidak disentuh).
subprocess.run("pkill -f '[o]llama serve'; pkill -f '[l]lm_auth_proxy'; pkill -f '[c]f-llm tunnel'",
               shell=True)
if shutil.which("ollama") is None:
    !apt-get -qq install -y zstd > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh > /dev/null

ollama_env = dict(os.environ,
                  OLLAMA_HOST="127.0.0.1:11434",  # tidak pernah dibuka langsung ke tunnel
                  OLLAMA_KEEP_ALIVE="-1",         # model tetap di GPU, tidak bongkar-muat per pesan
                  OLLAMA_NUM_PARALLEL="1",        # satu jawaban sekaligus: memori GPU terbatas
                  OLLAMA_CONTEXT_LENGTH="4096")   # cukup untuk prompt bot; memori tetap terduga
ollama = subprocess.Popen(["ollama", "serve"], env=ollama_env,
                          stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama tidak mau start; lihat ollama.log.")
subprocess.run(["ollama", "pull", MODEL], check=True, env=ollama_env, stdout=subprocess.DEVNULL)
print("Model siap:", MODEL)

In [ ]:
%%writefile llm_auth_proxy.py
"""Puts an API key in front of Ollama, which has no authentication of its own."""
import hmac
import os
import sys
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

KEY = os.environ["LLM_API_KEY"].encode()
UPSTREAM = os.environ.get("LLM_UPSTREAM", "http://127.0.0.1:11434")
# Only what the bot uses. Model management (/api/pull, /api/delete, ...) stays unreachable.
ALLOWED = {("POST", "/v1/chat/completions"), ("GET", "/v1/models")}
MAX_BODY = 256 * 1024


class Proxy(BaseHTTPRequestHandler):
    def reply(self, status, body, content_type="application/json"):
        self.send_response(status)
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def handle_request(self):
        path = self.path.split("?", 1)[0]
        if (self.command, path) not in ALLOWED:
            return self.reply(404, b'{"error":"not_found"}')
        if not hmac.compare_digest(self.headers.get("Authorization", "").encode(), b"Bearer " + KEY):
            return self.reply(401, b'{"error":"unauthorized"}')
        try:
            length = int(self.headers.get("Content-Length") or 0)
        except ValueError:
            return self.reply(400, b'{"error":"bad_length"}')
        if length < 0 or length > MAX_BODY:
            return self.reply(413, b'{"error":"too_large"}')
        data = self.rfile.read(length) if length else None
        request = urllib.request.Request(UPSTREAM + path, data=data, method=self.command,
                                         headers={"Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(request, timeout=300) as upstream:
                self.reply(upstream.status, upstream.read(),
                           upstream.headers.get("Content-Type", "application/json"))
        except urllib.error.HTTPError as error:
            self.reply(error.code, error.read())
        except Exception:
            self.reply(502, b'{"error":"upstream_unavailable"}')

    do_GET = handle_request
    do_POST = handle_request

    def log_request(self, code="-", size="-"):
        # Method, path and status only: never the key or the message text.
        sys.stderr.write("%s %s %s\n" % (self.command, self.path.split("?", 1)[0], code))


port = int(sys.argv[1]) if len(sys.argv) > 1 else 8081
ThreadingHTTPServer(("127.0.0.1", port), Proxy).serve_forever()

In [ ]:
import json, re, secrets
os.environ["WA_LLM_API_KEY"] = secrets.token_urlsafe(24)  # key baru tiap run, kecuali diset di Secrets
try:  # opsional: key tetap -> Colab Secrets "WA_LLM_API_KEY", isi juga di apps/whatsapp-bot/.env
    from google.colab import userdata
    os.environ["WA_LLM_API_KEY"] = userdata.get("WA_LLM_API_KEY")
except Exception:
    pass
proxy = subprocess.Popen(["python", "llm_auth_proxy.py", str(LLM_PORT)],
                         env=dict(os.environ, LLM_API_KEY=os.environ["WA_LLM_API_KEY"]),
                         stdout=open("proxy.log", "w"), stderr=subprocess.STDOUT)

def ask(base_url, question, key):
    """Same request shape as the bot (src/chat/assistant.ts). Returns (status, answer, seconds)."""
    payload = {"model": MODEL, "stream": False, "temperature": 0.2, "max_tokens": 300,
               "messages": [{"role": "user", "content": question + " /no_think"}]}
    headers = {"Content-Type": "application/json"}
    if key:
        headers["Authorization"] = "Bearer " + key
    request = urllib.request.Request(base_url + "/chat/completions", data=json.dumps(payload).encode(),
                                     headers=headers, method="POST")
    start = time.time()
    try:
        with urllib.request.urlopen(request, timeout=300) as response:
            body = json.load(response)
            status = response.status
    except urllib.error.HTTPError as error:
        return error.code, None, time.time() - start
    content = body["choices"][0]["message"].get("content") or ""
    answer = re.sub(r"<think>[\s\S]*?</think>", "", content).strip()  # sama seperti bot
    return status, answer, time.time() - start

import urllib.error
local = f"http://127.0.0.1:{LLM_PORT}/v1"
for _ in range(30):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{LLM_PORT}/v1/models", timeout=2)
    except urllib.error.HTTPError:
        break  # proxy hidup (menolak tanpa key, sesuai harapan)
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Proxy tidak mau start; lihat proxy.log.")

# Permintaan pertama memuat model ke GPU, jadi lebih lama dari biasanya.
status, answer, seconds = ask(local, "Jawab dengan satu kalimat: apa itu ayam petelur?",
                              os.environ["WA_LLM_API_KEY"])
print(f"Pemanasan: HTTP {status}, {seconds:.1f} dtk -> {answer!r}\n")
ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True, env=ollama_env).stdout
print(ps)
if "100% GPU" not in ps:
    print("PERINGATAN: model tidak sepenuhnya di GPU (lihat kolom PROCESSOR). Jawaban akan lambat.\n"
          "Pakai model lebih kecil, atau JALANKAN_VLM = False.")
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cf-llm && chmod +x cf-llm
import threading
llm_tunnel = subprocess.Popen(["./cf-llm", "tunnel", "--url", f"http://localhost:{LLM_PORT}"],
                              stderr=subprocess.PIPE, text=True)
llm_url = None
while llm_url is None:
    line = llm_tunnel.stderr.readline()
    if not line and llm_tunnel.poll() is not None:
        raise RuntimeError("cloudflared berhenti sebelum memberi URL.")
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        llm_url = match.group(0)
# Terus baca log cloudflared; pipe yang penuh bisa membuat tunnel macet.
threading.Thread(target=lambda: [None for _ in llm_tunnel.stderr], daemon=True).start()

# Uji lewat tunnel, persis seperti bot: tanpa key harus ditolak, dengan key harus dijawab.
public = llm_url + "/v1"
for _ in range(20):  # URL trycloudflare butuh beberapa detik sampai aktif
    try:
        no_key = ask(public, "tes", None)[0]
        break
    except Exception:
        time.sleep(3)
else:
    raise RuntimeError("Tunnel belum bisa dihubungi; jalankan ulang sel ini.")
status, answer, seconds = ask(public, "Suhu kandang 28,5 °C. Sebutkan suhunya dalam satu kalimat.",
                              os.environ["WA_LLM_API_KEY"])
print(f"Tanpa key: HTTP {no_key} (harus 401)")
print(f"Dengan key: HTTP {status}, {seconds:.1f} dtk -> {answer!r}")

print("\nSalin ke apps/whatsapp-bot/.env, lalu restart bot:\n")
print("WA_LLM_BASE_URL=" + public)
print("WA_LLM_MODEL=" + MODEL)
print("WA_LLM_API_KEY=" + os.environ["WA_LLM_API_KEY"])
print("WA_LLM_TIMEOUT_MS=60000")
print("\nJangan tutup tab ini selama bot dipakai. URL dan key berubah setiap Run all.")

## Jika ada masalah

- **`PERINGATAN: model tidak sepenuhnya di GPU`**: memori GPU tidak cukup. Pakai `MODEL = "qwen3:4b"`
  (bukan 8b), atau `JALANKAN_VLM = False`.
- **Bot selalu membalas ringkasan status, bukan jawaban LLM**: cek log bot. `401` berarti
  `WA_LLM_API_KEY` di `.env` tidak sama dengan output di atas; gagal koneksi berarti sesi Colab mati
  atau URL sudah berganti.
- **Log**: `ollama.log` dan `proxy.log` di panel Files (folder `/content`).